In [ ]:
# !pip uninstall -y numpy matplotlib scikit-learn imbalanced-learn
# !pip install numpy==1.23.5 matplotlib==3.6.3 scikit-learn==1.1.3 imbalanced-learn==0.10.1

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: matplotlib 3.7.2
Uninstalling matplotlib-3.7.2:
  Successfully uninstalled matplotlib-3.7.2
Found existing installation: scikit-learn 1.2.2
Uninstalling scikit-learn-1.2.2:
  Successfully uninstalled scikit-learn-1.2.2
Found existing installation: imbalanced-learn 0.13.0
Uninstalling imbalanced-learn-0.13.0:
  Successfully uninstalled imbalanced-learn-0.13.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 101.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.0/32.0 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.0/226.0 kB 16.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 

In [ ]:
import os
import time
import pickle

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.fftpack import dct

from skimage.feature import local_binary_pattern

from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix

from imblearn.over_sampling import SMOTE

In [ ]:


# Replace 'your_file.csv' with the actual path to your CSV file
file_path = r"/kaggle/input/auitd-dataset/Labels_AUITD.csv"

column_names = ['Image', 'Stage_1', 'Stage_2']

df = pd.read_csv(file_path, header=None, names=column_names)

df=df[1:792]

print(df)

     Image Stage_1 Stage_2
1    2_100       1       2
2    2_101       1       2
3    2_102       1       2
4    2_103       1       2
5    2_104       1       2
..     ...     ...     ...
787   5_95       2       5
788   5_96       2       5
789   5_97       2       5
790   5_98       2       5
791   5_99       2       5

[791 rows x 3 columns]


In [ ]:


# Define the root directory
rootdir = r"/kaggle/input/auitd-dataset/All_images/All_images"
image_path = [os.path.join(rootdir, file) for file in os.listdir(rootdir)]

# Load your labels DataFrame (REPLACE WITH YOUR ACTUAL PATH)
# df = pd.read_csv('path_to_labels.csv')

# LBP parameters
radius = 1
n_points = 8 * radius
method = 'uniform'  # Uniform LBP [2][5]
cell_size = 16      # 16x16 pixel cells


# Precompute uniform patterns mapping (like skimage does)
def uniform_lookup_table(P=8):
    table = np.zeros(2**P, dtype=np.int32)
    index = 0
    for i in range(2**P):
        # binary string for pattern
        b = [(i >> j) & 1 for j in range(P)]
        # count transitions (0->1 or 1->0 circular)
        transitions = sum(b[k] != b[(k+1) % P] for k in range(P))
        if transitions <= 2:  # uniform pattern
            table[i] = index
            index += 1
        else:
            table[i] = 58  # non-uniform bin
    return table

uniform_map = uniform_lookup_table(P=n_points)

def mean_lbp(gray_image, radius=1):
    h, w = gray_image.shape
    lbp_image = np.zeros((h, w), dtype=np.uint8)
    offsets = [(-1, -1), (-1, 0), (-1, 1),
               ( 0, 1), ( 1, 1), ( 1, 0),
               ( 1,-1), ( 0,-1)]  # standard 8 neighbors clockwise

    for i in range(radius, h-radius):
        for j in range(radius, w-radius):
            neighbors = [gray_image[i+dy, j+dx] for dy, dx in offsets]

            # Mean of 9 pixels (center + neighbors)
            patch = gray_image[i-radius:i+radius+1, j-radius:j+radius+1]
            mean_val = np.mean(patch)

            # Compare neighbors with mean
            binary = (np.array(neighbors) >= mean_val).astype(int)

            # Convert to decimal code
            code = 0
            for bit in binary:
                code = (code << 1) | bit

            # Map to uniform bin
            lbp_image[i, j] = uniform_map[code]

    return lbp_image

def extract_mean_lbp_descriptors(image):
    # Resize and convert to grayscale
    resized_image = cv2.resize(image, (256, 256))
    gray_image = cv2.cvtColor(resized_image, cv2.COLOR_BGR2GRAY)

    # Compute Mean-LBP with uniform mapping
    lbp_image = mean_lbp(gray_image, radius)

    # Initialize feature vector
    feature_vector = []

    # Process each 16x16 cell
    for y in range(0, 256, cell_size):
        for x in range(0, 256, cell_size):
            cell = lbp_image[y:y+cell_size, x:x+cell_size]

            # Histogram with 59 bins (58 uniform + 1 non-uniform)
            hist, _ = np.histogram(cell.ravel(), bins=59, range=(0, 58))

            # Normalize histogram
            hist = hist.astype("float")
            hist /= (hist.sum() + 1e-7)

            feature_vector.extend(hist)

    return np.array(feature_vector)

# Initialize DataFrame for storing image names, features, and labels
lbp_df = pd.DataFrame(columns=['image_name', 'features', 'labels_1', 'labels_2'])

def main():
    global lbp_df
    t0 = time.time()
    
    for file in image_path:
        # Read image
        img = cv2.imread(file)
        if img is None:
            print(f"Warning: Failed to read {file}")
            continue
            
        # Get image name
        img_name = os.path.basename(file).split('.')[0]
        
        # Verify image exists in labels DataFrame
        if img_name not in df['Image'].values:
            print(f"Warning: {img_name} not found in labels DataFrame")
            continue
            
        # Get labels
        label_1 = df.loc[df['Image'] == img_name, 'Stage_1'].iloc[0]
        label_2 = df.loc[df['Image'] == img_name, 'Stage_2'].iloc[0]
        
        # Extract features
        img_features = extract_mean_lbp_descriptors(img)
        
        # Store in DataFrame
        new_entry = pd.DataFrame({
            'image_name': [img_name],
            'features': [img_features],
            'labels_1': [label_1],
            'labels_2': [label_2]
        })
        
        lbp_df = pd.concat([lbp_df, new_entry], ignore_index=True)
    
    print(f'Time taken: {time.time()-t0:.2f} seconds')
    print('\nFeature extraction complete.')

# Execute main function
if __name__ == "__main__":
    main()

# Final DataFrame output
print(lbp_df)

# To check feature length:
print("Feature vector length (bins):", len(lbp_df['features'].iloc[-1]))

Time taken: 795.56 seconds

Feature extraction complete.
    image_name                                           features labels_1  \
0         5_59  [0.12109374995269774, 0.0, 0.00781249999694824...        2   
1        4_316  [0.12109374995269774, 0.0, 0.0, 0.0, 0.0, 0.0,...        2   
2        4_122  [0.12109374995269774, 0.0, 0.00781249999694824...        2   
3        4_388  [0.12109374995269774, 0.0, 0.0, 0.0, 0.0, 0.0,...        2   
4         4_77  [0.12109374995269774, 0.0, 0.00390624999847412...        2   
..         ...                                                ...      ...   
776       3_87  [0.12109374995269774, 0.0, 0.0, 0.0, 0.0, 0.0,...        1   
777       2_87  [0.12109374995269774, 0.0, 0.0, 0.0, 0.0, 0.0,...        1   
778       3_62  [0.12109374995269774, 0.0039062499984741208, 0...        1   
779       5_80  [0.12109374995269774, 0.0, 0.0, 0.007812499996...        2   
780       2_77  [0.12109374995269774, 0.0, 0.00390624999847412...        1   

    la

In [7]:
print("Length of features in hog df", len(lbp_df['features'].iloc[-1]))


Length of features in hog df 15104


In [8]:
lbp_df

,image_name,features,labels_1,labels_2
0,5_59,"[0.12109374995269774, 0.0, 0.00781249999694824...",2,5
1,4_316,"[0.12109374995269774, 0.0, 0.0, 0.0, 0.0, 0.0,...",2,4
2,4_122,"[0.12109374995269774, 0.0, 0.00781249999694824...",2,4
3,4_388,"[0.12109374995269774, 0.0, 0.0, 0.0, 0.0, 0.0,...",2,4
4,4_77,"[0.12109374995269774, 0.0, 0.00390624999847412...",2,4
...,...,...,...,...
776,3_87,"[0.12109374995269774, 0.0, 0.0, 0.0, 0.0, 0.0,...",1,3
777,2_87,"[0.12109374995269774, 0.0, 0.0, 0.0, 0.0, 0.0,...",1,2
778,3_62,"[0.12109374995269774, 0.0039062499984741208, 0...",1,3
779,5_80,"[0.12109374995269774, 0.0, 0.0, 0.007812499996...",2,5


In [9]:
# Dropping the 'labels_2' column to get Stage 1 data
lbp_df_stage_1 = lbp_df.drop(columns=['labels_2'])

# Dropping the 'labels_1' column to get Stage 2 data
lbp_df_stage_2 = lbp_df.drop(columns=['labels_1'])


In [10]:
# Display the updated DataFrame
print(lbp_df_stage_1)


    image_name                                           features labels_1
0         5_59  [0.12109374995269774, 0.0, 0.00781249999694824...        2
1        4_316  [0.12109374995269774, 0.0, 0.0, 0.0, 0.0, 0.0,...        2
2        4_122  [0.12109374995269774, 0.0, 0.00781249999694824...        2
3        4_388  [0.12109374995269774, 0.0, 0.0, 0.0, 0.0, 0.0,...        2
4         4_77  [0.12109374995269774, 0.0, 0.00390624999847412...        2
..         ...                                                ...      ...
776       3_87  [0.12109374995269774, 0.0, 0.0, 0.0, 0.0, 0.0,...        1
777       2_87  [0.12109374995269774, 0.0, 0.0, 0.0, 0.0, 0.0,...        1
778       3_62  [0.12109374995269774, 0.0039062499984741208, 0...        1
779       5_80  [0.12109374995269774, 0.0, 0.0, 0.007812499996...        2
780       2_77  [0.12109374995269774, 0.0, 0.00390624999847412...        1

[781 rows x 3 columns]


In [11]:
print(lbp_df_stage_2)


    image_name                                           features labels_2
0         5_59  [0.12109374995269774, 0.0, 0.00781249999694824...        5
1        4_316  [0.12109374995269774, 0.0, 0.0, 0.0, 0.0, 0.0,...        4
2        4_122  [0.12109374995269774, 0.0, 0.00781249999694824...        4
3        4_388  [0.12109374995269774, 0.0, 0.0, 0.0, 0.0, 0.0,...        4
4         4_77  [0.12109374995269774, 0.0, 0.00390624999847412...        4
..         ...                                                ...      ...
776       3_87  [0.12109374995269774, 0.0, 0.0, 0.0, 0.0, 0.0,...        3
777       2_87  [0.12109374995269774, 0.0, 0.0, 0.0, 0.0, 0.0,...        2
778       3_62  [0.12109374995269774, 0.0039062499984741208, 0...        3
779       5_80  [0.12109374995269774, 0.0, 0.0, 0.007812499996...        5
780       2_77  [0.12109374995269774, 0.0, 0.00390624999847412...        2

[781 rows x 3 columns]


In [ ]:


# Convert labels_2 to numeric if it's not already
lbp_df_stage_2['labels_2'] = pd.to_numeric(lbp_df_stage_2['labels_2'])

# Filter the dataframe to include only rows with labels_2 values 2 or 3
lbp_df_benign = lbp_df_stage_2[lbp_df_stage_2['labels_2'].isin([2, 3])]

# Display the filtered dataframe
print(lbp_df_benign)


    image_name                                           features  labels_2
5         3_95  [0.12109374995269774, 0.0, 0.0, 0.0, 0.0, 0.0,...         3
9         2_57  [0.12109374995269774, 0.0, 0.00390624999847412...         2
14        3_77  [0.12109374995269774, 0.0, 0.0, 0.0, 0.0, 0.0,...         3
16        3_42  [0.12109374995269774, 0.0, 0.0, 0.003906249998...         3
17        3_43  [0.12109374995269774, 0.0, 0.0, 0.0, 0.0, 0.00...         3
..         ...                                                ...       ...
775      2_110  [0.12109374995269774, 0.0, 0.0, 0.0, 0.0, 0.0,...         2
776       3_87  [0.12109374995269774, 0.0, 0.0, 0.0, 0.0, 0.0,...         3
777       2_87  [0.12109374995269774, 0.0, 0.0, 0.0, 0.0, 0.0,...         2
778       3_62  [0.12109374995269774, 0.0039062499984741208, 0...         3
780       2_77  [0.12109374995269774, 0.0, 0.00390624999847412...         2

[182 rows x 3 columns]


In [13]:
import pandas as pd

# Convert labels_2 to numeric if it's not already
lbp_df_stage_2['labels_2'] = pd.to_numeric(lbp_df_stage_2['labels_2'])

# Filter the dataframe to include only rows with labels_2 values 4 or 5 (malignant)
lbp_df_malignant = lbp_df_stage_2[lbp_df_stage_2['labels_2'].isin([4, 5])]

# Display the filtered dataframe
print(lbp_df_malignant)


    image_name                                           features  labels_2
0         5_59  [0.12109374995269774, 0.0, 0.00781249999694824...         5
1        4_316  [0.12109374995269774, 0.0, 0.0, 0.0, 0.0, 0.0,...         4
2        4_122  [0.12109374995269774, 0.0, 0.00781249999694824...         4
3        4_388  [0.12109374995269774, 0.0, 0.0, 0.0, 0.0, 0.0,...         4
4         4_77  [0.12109374995269774, 0.0, 0.00390624999847412...         4
..         ...                                                ...       ...
767       4_13  [0.12109374995269774, 0.0, 0.00390624999847412...         4
769      4_493  [0.12109374995269774, 0.0, 0.00390624999847412...         4
771      4_156  [0.12109374995269774, 0.0, 0.01171874999542236...         4
772      4_248  [0.12109374995269774, 0.0, 0.0, 0.0, 0.0, 0.0,...         4
779       5_80  [0.12109374995269774, 0.0, 0.0, 0.007812499996...         5

[599 rows x 3 columns]


# Stage2 Malignant(TIRADS4/TIRADS5)

In [ ]:

# =====================================================
# Ensure labels are int
# =====================================================

lbp_df_malignant['labels_2'] = (
    lbp_df_malignant['labels_2'].astype(int)
)

# =====================================================
# Separate features, labels, names
# =====================================================

X = np.array(
    lbp_df_malignant['features'].tolist()
)

y = np.array(
    lbp_df_malignant['labels_2']
)

names = np.array(
    lbp_df_malignant['image_name']
)

output_root = "/kaggle/working"

model_dir = os.path.join(
    output_root,
    "models_stage_2"
)

result_dir = os.path.join(
    output_root,
    "results_stage_2"
)

os.makedirs(model_dir, exist_ok=True)
os.makedirs(result_dir, exist_ok=True)

# =====================================================
# Parameters
# =====================================================

num_splits = 5
kernel = 'rbf'

# =====================================================
# Metric storage
# =====================================================

accuracy_scores = []
specificity_scores = []
sensitivity_scores = []
precision_scores = []
f1_scores = []

# =====================================================
# Stratified KFold
# =====================================================

kfold = StratifiedKFold(
    n_splits=num_splits,
    shuffle=True
)

# =====================================================
# Store predictions
# =====================================================

all_results = pd.DataFrame(
    columns=[
        'fold',
        'image_name',
        'true_label',
        'predicted_label'
    ]
)

# =====================================================
# Cross-validation
# =====================================================

for fold, (train_index, test_index) in enumerate(
        kfold.split(X, y), start=1):

    print("\n" + "="*50)
    print(f"Fold {fold}")
    print("="*50)

    # =================================================
    # Split data
    # =================================================

    X_train = X[train_index]
    X_test  = X[test_index]

    y_train = y[train_index]
    y_test  = y[test_index]

    names_test = names[test_index]

    # =================================================
    # Original Test Distribution
    # =================================================

    unique_test, counts_test = np.unique(
        y_test,
        return_counts=True
    )

    print("\nOriginal Test Distribution:")

    for cls, cnt in zip(unique_test, counts_test):
        print(f"Class {cls}: {cnt}")

    print(f"\nTotal Test Samples: {len(y_test)}")

    # =================================================
    # SMOTE on TRAIN
    # =================================================

    smote_train = SMOTE(random_state=5)

    X_train_smote, y_train_smote = (
        smote_train.fit_resample(
            X_train,
            y_train
        )
    )

    # =================================================
    # Training Distribution After SMOTE
    # =================================================

    print("\nTraining Distribution After SMOTE:")

    unique_sm, counts_sm = np.unique(
        y_train_smote,
        return_counts=True
    )

    for cls, cnt in zip(unique_sm, counts_sm):
        print(f"Class {cls}: {cnt}")

    X_test_final = X_test
    y_test_final = y_test

    # =================================================
    # TRAIN SVM
    # =================================================

    svm = SVC(
        kernel=kernel
    
    )

    svm.fit(
        X_train_smote,
        y_train_smote
    )

    # =================================================
    # PREDICT
    # =================================================

    y_pred = svm.predict(
        X_test_final
    )

    # =================================================
    # Save model
    # =================================================

    model_path = os.path.join(
        model_dir,
        f"svm_fold_{fold}.pkl"
    )

    with open(model_path, 'wb') as f:
        pickle.dump(svm, f)

    # =================================================
    # Save fold predictions
    # =================================================

    fold_df = pd.DataFrame({
        'fold': [fold] * len(y_test_final),
        'image_name': ['mixed_test']
                      * len(y_test_final),
        'true_label': y_test_final,
        'predicted_label': y_pred
    })

    all_results = pd.concat(
        [all_results, fold_df],
        ignore_index=True
    )

    # =================================================
    # Confusion Matrix
    # =================================================

    cm = confusion_matrix(
        y_test_final,
        y_pred,
    )

    tn, fp, fn, tp = cm.ravel()

    # =================================================
    # Metrics
    # =================================================

    specificity = (
        tn / (tn + fp + 1e-10)
    )

    sensitivity = (
        tp / (tp + fn + 1e-10)
    )

    accuracy = (
        (tp + tn)
        / (tp + tn + fp + fn + 1e-10)
    )

    precision = (
        tp / (tp + fp + 1e-10)
    )

    f1_score = (
        2 * (precision * sensitivity)
    ) / (
        precision + sensitivity + 1e-10
    )

    # =================================================
    # Store metrics
    # =================================================

    specificity_scores.append(
        specificity
    )

    sensitivity_scores.append(
        sensitivity
    )

    accuracy_scores.append(
        accuracy
    )

    precision_scores.append(
        precision
    )

    f1_scores.append(
        f1_score
    )

# =====================================================
# SAVE CSV
# =====================================================

csv_path = os.path.join(
    result_dir,
    "fold_predictions.csv"
)

all_results.to_csv(
    csv_path,
    index=False
)

# =====================================================
# Average Metrics
# =====================================================

average_accuracy = np.mean(
    accuracy_scores
)

average_specificity = np.mean(
    specificity_scores
)

average_sensitivity = np.mean(
    sensitivity_scores
)

average_precision = np.mean(
    precision_scores
)

average_f1_score = np.mean(
    f1_scores
)

# =====================================================
# FINAL RESULTS
# =====================================================

print("\n" + "="*50)
print("FINAL RESULTS")
print("="*50)

print("\nAverage Precision:",
      average_precision)

print("Average F1 Score:",
      average_f1_score)

print("\nAverage Specificity:",
      average_specificity)

print("Average Sensitivity (Recall):",
      average_sensitivity)

print("Average Accuracy:",
      average_accuracy)


Fold 1

Original Test Distribution:
Class 4: 101
Class 5: 19

Total Test Samples: 120


/tmp/ipykernel_19/2358878035.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lbp_df_malignant['labels_2'] = (
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenM


Training Distribution After SMOTE:
Class 4: 404
Class 5: 404

Fold 2

Original Test Distribution:
Class 4: 101
Class 5: 19

Total Test Samples: 120


OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detec


Training Distribution After SMOTE:
Class 4: 404
Class 5: 404

Fold 3

Original Test Distribution:
Class 4: 101
Class 5: 19

Total Test Samples: 120


...


Training Distribution After SMOTE:
Class 4: 404
Class 5: 404

Fold 4

Original Test Distribution:
Class 4: 101
Class 5: 19

Total Test Samples: 120


OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detec


Training Distribution After SMOTE:
Class 4: 404
Class 5: 404

Fold 5

Original Test Distribution:
Class 4: 101
Class 5: 18

Total Test Samples: 119


OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detec


Training Distribution After SMOTE:
Class 4: 404
Class 5: 404

FINAL RESULTS

Average Precision: 0.9096105174123743
Average F1 Score: 0.8161886778223405

Average Specificity: 0.9267326732664092
Average Sensitivity (Recall): 0.7405940594052074
Average Accuracy: 0.8336633663362211
